In [1]:
import pandas as pd
import requests
import re
import time
from bs4 import BeautifulSoup as bs
from dbio3 import to_db, load_data

# 뉴스목록, 날짜, id수집

In [3]:
url1 = "https://fintech.or.kr/web/board/boardContentsList.do"

In [ ]:
miv_pageNo=2&miv_pageSize=&total_cnt=&LISTOP=&mode=W&contents_id=&board_id=6&p_reg_userno=

In [4]:
payload = dict(miv_pageNo=1, mode='W', board_id=6)

In [5]:
r1 = requests.get(url1, params=payload)
print(r1.status_code)
soup = bs(r1.content, "lxml")
soup

200


<html><head><script language="javascript">
$(document).ready(function(){

	// 댓글수 노출여부 확인후 숨기기(미구현)
	
		$(".comment_cnt").hide();
	

	// 답변 노출여부 확인후 숨기기(미구현)
	
		$(".reply_status").hide();
	

	// 모바일 전용 페이지 숨기지
	//$(".mobile_list").hide();

	$(function() {

	});

	$("#searchtxt").keydown(function(key) {
		if (key.keyCode == 13) {
			search();
		}
	});
});
</script>
</head><body><div class="boardlist_top">
<ul class="box">
<li class="select">
<div class="optionbox">
<select id="searchkey" name="searchkey">
<option value="I">전체</option>
<option value="T">제목</option>
<option value="C">내용</option>
</select>
</div>
</li>
<li class="input">
<div class="inpbox"><input class="txt" id="searchtxt" name="searchtxt" placeholder="검색어 입력" title="검색어 입력" type="text" value=""/></div>
</li>
<li class="button">
<button class="btn3 bg_blue" onclick="search();" title="검색" type="button">검색</button>
<!-- <button type="button" class="btn3 bg_gray" title="재검색">재검색</button> -->
</li>
</ul>
</div>
<!--// list_t

mobile용

In [6]:
len(soup.select("ul.list > li > a"))

15

In [7]:
# 제목과 뉴스 id 
soup.select("a.txtl")

[<a class="txtl" href="javascript:contentsView('bf1b44ca61d74074a455cf21e180451c')">[경기도] 미래콘텐츠 성과공유회(AXR)에 초대합니다. (11/20(목) ~ 21(금))</a>,
 <a class="txtl" href="javascript:contentsView('d4ad01f29e27456bacd3bf33c0668232')">[우리금융그룹] 2026 디노랩 서울 7기/부산 2기/경남 3기 모집(~12/03)</a>,
 <a class="txtl" href="javascript:contentsView('b4b0b17c318d45158645c81c1472e29a')">[금융보안원] 「데이터허브」 이용 안내</a>,
 <a class="txtl" href="javascript:contentsView('51cb0e37c4064f2a985436e2ae3378df')">[국무조정실] 규제개혁신문고 제도 안내</a>,
 <a class="txtl" href="javascript:contentsView('3c24d05040984c79a7cdfd2c921952eb')">[기술보증기금] 핀테크 기업을 위한 주요 보증상품 안내</a>,
 <a class="txtl" href="javascript:contentsView('25f4360d5b9847a996828099e8d52ade')">[신용보증기금] 스타트업 지원 업무 안내</a>,
 <a class="txtl" href="javascript:contentsView('9c16609a99ad4c55914ffa9a7f2d58b7')">[한국무역보험공사] K-Sure 수출컨설팅 사업 안내</a>,
 <a class="txtl" href="javascript:contentsView('6b14460b5adc480aa050c4c535af8b72')">[한국예탁결제원] 예탁결제원 보유 데이터 활용 안내</a>,
 <a class="txtl" href="javascript:

web용은 Table 안에

In [8]:
# 제목과 뉴스 id 
soup.select("a.txtl")[0]['href'].split("'")[1]

'bf1b44ca61d74074a455cf21e180451c'

In [9]:
# 날짜
soup.select("td.last")[0].text.strip()

'2025-11-12'

# news list id와 날짜 모으기

In [10]:
contents_id_list = []
for date, content_list in zip(soup.select("td.last"), soup.select("a.txtl")):
    date = date.text.strip()
    contents_id = content_list['href'].split("'")[1]
    contents_id_list.append((date, contents_id))
contents_id_list

[('2025-11-12', 'bf1b44ca61d74074a455cf21e180451c'),
 ('2025-11-12', 'd4ad01f29e27456bacd3bf33c0668232'),
 ('2025-09-17', 'b4b0b17c318d45158645c81c1472e29a'),
 ('2024-02-20', '51cb0e37c4064f2a985436e2ae3378df'),
 ('2022-12-19', '3c24d05040984c79a7cdfd2c921952eb'),
 ('2022-12-19', '25f4360d5b9847a996828099e8d52ade'),
 ('2022-06-20', '9c16609a99ad4c55914ffa9a7f2d58b7'),
 ('2021-12-03', '6b14460b5adc480aa050c4c535af8b72'),
 ('2021-04-07', 'd4cbbeb3f9b3444cbf1856b48f9f2926'),
 ('2025-11-27', '62ec460f84a24e54bb51311c81fdfb1c'),
 ('2025-11-26', '39547f2a0ee34a5097ff4e92f6df9f6e'),
 ('2025-11-25', '933a27b8c60a4499957e17ac74649983'),
 ('2025-11-24', '084feb7185844e6fa6d7ff981c514de0'),
 ('2025-11-21', 'edd05197dbf749118cd7d16918ab3cc4'),
 ('2025-11-20', '12c003b7478b4c359c558ecc1edcc342')]

In [11]:
# newslist의 url
url2 = "https://fintech.or.kr/web/board/boardContentsView.do"

In [12]:
# news list의 payload
payload2 = dict(miv_pageNo=1, mode="W", contents_id="39547f2a0ee34a5097ff4e92f6df9f6e", board_id=6, searchkey="I")

In [13]:
r2 = requests.get(url2, params=payload2)
print(r2.status_code)
soup2 = bs(r2.content,'lxml')
soup2

200


<!DOCTYPE html>
<html lang="ko" xml:lang="ko">
<head>
<title>
		
		
		핀테크 포털 - 한국핀테크지원센터
		
		</title>
<meta content="핀테크 생태계 활성화로 금융의 혁신과 성장을 지원합니다.." name="description"/>
<meta content="핀테크 포털 - 한국핀테크지원센터" name="keywords"/>
<meta content="http://fintech.or.kr/images/preview_page.png" property="og:image"/>
<meta content="text/html; charset=utf-8" http-equiv="Content-Type"/>
<meta content="width=device-width,initial-scale=1.0,maximum-scale=1.0,minimum-scale=1.0,user-scalable=no" name="viewport"/>
<meta content="XpressEngine" name="Generator"/>
<meta content="IE=edge" http-equiv="X-UA-Compatible"/>
<meta content="o8yb702i79l2intg1e9ov2hyavzbji" name="facebook-domain-verification"/>
<!-- <link rel="shortcut icon" href="/images/web/favicon.ico" type="image/x-icon"> -->
<!-- <link rel="icon" href="/images/web/favicon.ico" type="image/x-icon"> -->
<link href="/css/web/default.css?ver=20200916" rel="stylesheet" type="text/css"/>
<link href="/css/web/reset.css?ver=20210629" rel="stylesheet" t

In [14]:
# 뉴스 리스트 목록
soup2.select("tbody td a")[0].text

"[한국핀테크지원센터X구름] K-디지털트레이닝 '핀테크 인턴십 코스' 4기 훈련생 모집(~12.8.)"

In [15]:
soup2.select("tbody td a")[0]['href']

'https://fintech.or.kr/web/board/boardContentsView.do?board_id=3&contents_id=ba9a817cc8434910a4c6f47ff2e19099&menu_id=6300'

In [16]:
soup2.select("tbody td a")[6]['href']

'https://n.news.naver.com/mnews/article/003/0013620831?sid=101'

# 위에서 개별 작업한 코드 모으기

In [17]:
# 핀테크 뉴스 리스트, 날짜, content_id 수집
contents_id_list = []
for page in range(1, 20):
    url1 = "https://fintech.or.kr/web/board/boardContentsList.do"
    payload = dict(miv_pageNo=page, mode='W', board_id=6)
    r1 = requests.post(url1, data=payload)
    print(r1.status_code)
    soup = bs(r1.content, "lxml")

    for date, content_list in zip(soup.select("td.last"), soup.select("a.txtl")):
        date = date.text.strip()
        contents_id = content_list['href'].split("'")[1]
        contents_id_list.append((date, contents_id))
    time.sleep(2)

result = {}
# 날짜별 뉴스 리스트의 세부 뉴스 제목, 링크
for idx, (date, content_id) in enumerate(contents_id_list):
    print(f"{idx+1}/{len(contents_id_list)} 수집중", end="\r")
    url2 = "https://fintech.or.kr/web/board/boardContentsView.do"
    payload2 = dict(miv_pageNo=1, mode="W", contents_id=content_id, board_id=6, searchkey="I")
    r2 = requests.get(url2, params=payload2)
#     print(r2.status_code)
    soup2 = bs(r2.content,'lxml')

    for td in soup2.select("tbody td a"): 
        if "https://n.news.naver.com" in td['href'] or "https://fintech.or.kr/web/" in td['href']:
            result.setdefault('date', []).append(date)
            result.setdefault('제목', []).append(td.text)
            result.setdefault('뉴스링크', []).append(td['href'])
            
df = pd.DataFrame(result)
df

200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200


,date,제목,뉴스링크
0,2025-11-27,[한국핀테크지원센터X구름] K-디지털트레이닝 '핀테크 인턴십 코스' 4기 훈련생 모...,https://fintech.or.kr/web/board/boardContentsV...
1,2025-11-27,[한국핀테크지원센터] (온라인) 핀테크를 통한 금융 AI 트렌드와 혁신 사례 교육생...,https://fintech.or.kr/web/board/boardContentsV...
2,2025-11-27,[한국핀테크지원센터] 2025년 핀테크기업 온라인 채용관 (사람인 saramin) ...,https://fintech.or.kr/web/board/boardContentsV...
3,2025-11-27,"금융위, 제7회 코리아 핀테크 위크 2025 개막",https://n.news.naver.com/mnews/article/277/000...
4,2025-11-27,"금융위, 금융공공데이터 162종 추가 개방… 상장사 정보·차보험 통계 등 포함",https://n.news.naver.com/mnews/article/366/000...
...,...,...,...
11769,2024-12-18,"中 고강도 부양책 발표 효과?…차이나 ETF 수익률 ""괜찮네""",https://n.news.naver.com/mnews/article/003/001...
11770,2024-12-18,구글 급소 `검색` 노린 오픈AI...AI발 검색 춘추전국 오나,https://n.news.naver.com/mnews/article/029/000...
11771,2024-12-18,"""日 혼다, 닛산과 합병 협상""…세계 3위 車업체 탄생하나",https://n.news.naver.com/mnews/article/277/000...
11772,2024-12-18,프랑스 이어 독일도 내각 붕괴… 유럽 리더십 위기,https://n.news.naver.com/mnews/article/005/000...


# 세부링크로 들어가서 뉴스 본문 수집

In [18]:
df2 = df[df['date'] > "2025-01-01"].copy()

In [19]:
df2['뉴스링크'][6]

'https://n.news.naver.com/mnews/article/003/0013621930?sid=101'

In [20]:
# 핀테크 지원센터 글
r3 = requests.get(df2['뉴스링크'][0])
soup3 = bs(r3.content, 'lxml')
soup3.select_one("div.content").text

'[한국핀테크지원센터x구름]「K-디지털 트레이닝 핀테크 인턴십 코스」4기 훈련생 모집(~12/8까지) \xa0  □ 프로그램 개요 - 국내 최대 핀테크 전문 지원기관인 한국핀테크지원센터와 국내 대기업(카카오, KT클라우드등) 부트캠프를 설계.운영해온 국내 대표 에듀테크 전문기관 구름(goorm)이 함께하는 핀테크 및 디지털 금융 산업 진출을\xa0목표로 하는 예비 실무자들을 위해 기획된 6개월 집중형 K-Digital Training 입니다.\ufeff \xa0  □ 신청 홈페이지 - (공식 신청 홈페이지) https://fintech-internship-course.goorm.io/ - (상세 안내 페이지)  https://www.notion.so/goormkdx/x-goorm-293c0ff4ce318024bea5ddcba23965d6 \xa0  □ 핀테크 인턴십 코스 과정 핵심 Point ① 핀테크 기업연계 현장형(인턴) 프로젝트 참여 : 한국핀테크지원센터의 협력기업과 현업 프로젝트 직접 참여 \xa0  ② 기업연계 매칭기업 서류전형 면제 : 프로젝트 진행 기업의 경우 채용전형 중 서류전형 면제 \xa0  ③ 우수 수료생 대상 인턴십 연계 : 수료자 중 우수 인재에게 협력기업 인턴십 또는 채용 연계 기회 제공 \xa0  ④ 핀테크 일자리 매칭 플랫폼 서비스 지원 : 수료 후 최대 2년간 취업 매칭, 채용 공고 연계, 1:1 커리어 컨설팅 제공 \xa0  ⑤ 현직자 멘토링 & 금융·핀테크 기업 및 기관 현장 견학 : 업계 선배 멘토링, 핀테크 기업 견학, 금융기관 방문 등 실무 인사이트 제공 \xa0  □ 주요 일정  구분일정비고서류접수2025.11.24(월) ~ 12.8(월)마감일 자정까지 서류 접수 가능발표2025.11.24(월) ~ 12.8(월)서류 제출 후 순차적 합격 통지교육기간2025.12.18(목) ~ 2026.06.18(금) 6개월 진행 (총 960시간) \xa0  □ 모집 과정 및 인원 과정명주요내용모집정원(명)핀테크 서비스 기획시장 분석, 

In [21]:
# 네이버뉴스 글
r3 = requests.get(df2['뉴스링크'][6])
soup3 = bs(r3.content, 'lxml')
soup3.select_one("#dic_area").text

'\n\n\n\n\n[서울=뉴시스]이지민 기자 = 금융감독원이 상장사 내부감사기구에 감사 품질 제고와 회계부정 방지를 위해 보다 적극적인 역할 수행에 나설 것을 주문했다. 금감원은 26일 서울 상장회사회관에서 상장사 감사위원·감사 및 유관기관 등과 \'내부감사기구 간담회\'를 개최했다.이날 간담회에 참석한 윤정숙 금감원 전문심의위원은 "신(新)외감법 시행 후 내부감사기구의 역할이 강화되며 회계투명성 확보의 실질적 주체로 자리 잡았다"며 "내부감사기구가 회계분식, 자금부정을 방지하기 위한 살아있는 내부통제의 핵심축으로 기능해야 한다"고 말했다. 이를 위해 ▲감사품질 중심의 외부감사인 선정 ▲내·외부감사인 간 유기적 협력 ▲절처한 내부통제시스템 감독 ▲독립성·전문성 확보 ▲회계부정 징후 발생 시 엄정 대응 등을 강조했다. 먼저, 감사품질을 높이기 위해 외부감사인 선정 시 독립성·전문성, 감사계획의 적정성, 투입시간의 충분성 등을 중점에 둘 것을 요청했다. 또 감사 과정에서 실제 투입시간, 인력 등을 점검해 감사 계획이 제대로 이행되는지 철저히 평가하도록 했다. 내·외부감사인 간의 긴밀한 협업도 강조했다. 경영진을 배제한 회의를 분기당 최소 1회 개최하고, 대면 회의를 통해 양방향으로 정보를 교류하는 등 실질적인 소통을 주문했다. 내부회계 평가에서는 통제 설계뿐 아니라 현장에서 통제가 제대로 작동하는지 확인하고, 미비점 발견 시 이사회에 충실히 보고하도록 했다.금감원은 내부감사기구에 독립된 보고체계 구축과 전문성 제고도 요청했다. 특히, 전담지원조직을 두고 임면 동의권과 직속 보고라인을 확보해 내부감사 기능을 강화해야 한다고 설명했다.아울러 회계부정이 발생하면 자체감사 또는 외부 전문가를 활용해 신속히 조사·시정하고, 결과를 금융당국과 감사인에게 제출해야 한다고도 했다. 내부감사기구는 조사 전 과정에 대한 감독의무를 지며, 주의의무 위반 시 책임을 질 수 있다고 경고했다.금감원 관계자는 "내부감사기구가 회계부정의 1차 방어선으로 기능할 수 있도록 제도·실무적 차원에

In [22]:
df2['뉴스본문'] = ""
df2

,date,제목,뉴스링크,뉴스본문
0,2025-11-27,[한국핀테크지원센터X구름] K-디지털트레이닝 '핀테크 인턴십 코스' 4기 훈련생 모...,https://fintech.or.kr/web/board/boardContentsV...,
1,2025-11-27,[한국핀테크지원센터] (온라인) 핀테크를 통한 금융 AI 트렌드와 혁신 사례 교육생...,https://fintech.or.kr/web/board/boardContentsV...,
2,2025-11-27,[한국핀테크지원센터] 2025년 핀테크기업 온라인 채용관 (사람인 saramin) ...,https://fintech.or.kr/web/board/boardContentsV...,
3,2025-11-27,"금융위, 제7회 코리아 핀테크 위크 2025 개막",https://n.news.naver.com/mnews/article/277/000...,
4,2025-11-27,"금융위, 금융공공데이터 162종 추가 개방… 상장사 정보·차보험 통계 등 포함",https://n.news.naver.com/mnews/article/366/000...,
...,...,...,...,...
11319,2025-01-02,"""당장 돈 안되고 성능향상 기대 못 미쳐도""… 세계는 AI인프라 `영끌`",https://n.news.naver.com/mnews/article/029/000...,
11320,2025-01-02,트럼프 2.0시대 중국…미 동맹국 이탈 기대하며 버티기[다시 만난 트럼프②],https://n.news.naver.com/mnews/article/032/000...,
11321,2025-01-02,'美 51번째 주' 모욕 당했는데…트럼프에 찍소리 못하는 이유 [김리안의 에네르기파...,https://n.news.naver.com/mnews/article/015/000...,
11322,2025-01-02,[산업은행] 2025년 상반기 KDB NextONE 참여 스타트업 모집(서울/부산)...,https://fintech.or.kr/web/board/boardContentsV...,


In [27]:
for idx, link in enumerate(df2['뉴스링크'][:10]):
    print(f"{idx+1}/{len(df2['뉴스링크'][:10])} 수집중", end="\r")
    r4 = requests.get(link)
    soup4 = bs(r4.content, 'lxml')
    if "https://fintech.or.kr/web/" in link:
        r3 = requests.get(link)
        soup3 = bs(r3.content, 'lxml')
        article = soup3.select_one("div.content").text
        df2.loc[idx, '뉴스본문'] = article
    elif "https://n.news.naver.com" in link:
        r3 = requests.get(link)
        soup3 = bs(r3.content, 'lxml')
        article = soup3.select_one("#dic_area").text
        df2.loc[idx, '뉴스본문'] = article
    to_db("fintech_news", "news_articles", df2.loc[[idx], :])
    time.sleep(3)
df2

OperationalError: (pymysql.err.OperationalError) (2003, "Can't connect to MySQL server on 'localhost' ([Errno 111] Connection refused)")
(Background on this error at: https://sqlalche.me/e/20/e3q8)

In [24]:
df2['뉴스본문']

0        [한국핀테크지원센터x구름]「K-디지털 트레이닝 핀테크 인턴십 코스」4기 훈련생 모집...
1                                                         
2                                                         
3                                                         
4                                                         
                               ...                        
11319                                                     
11320                                                     
11321                                                     
11322                                                     
11323                                                     
Name: 뉴스본문, Length: 11324, dtype: object

In [25]:
df2.to_csv("./data/fintech_news.csv")

In [26]:
df2 = pd.read_csv("./data/fintech_news.csv", index_col=0)
df2

,date,제목,뉴스링크,뉴스본문
0,2025-11-27,[한국핀테크지원센터X구름] K-디지털트레이닝 '핀테크 인턴십 코스' 4기 훈련생 모...,https://fintech.or.kr/web/board/boardContentsV...,[한국핀테크지원센터x구름]「K-디지털 트레이닝 핀테크 인턴십 코스」4기 훈련생 모집...
1,2025-11-27,[한국핀테크지원센터] (온라인) 핀테크를 통한 금융 AI 트렌드와 혁신 사례 교육생...,https://fintech.or.kr/web/board/boardContentsV...,NaN
2,2025-11-27,[한국핀테크지원센터] 2025년 핀테크기업 온라인 채용관 (사람인 saramin) ...,https://fintech.or.kr/web/board/boardContentsV...,NaN
3,2025-11-27,"금융위, 제7회 코리아 핀테크 위크 2025 개막",https://n.news.naver.com/mnews/article/277/000...,NaN
4,2025-11-27,"금융위, 금융공공데이터 162종 추가 개방… 상장사 정보·차보험 통계 등 포함",https://n.news.naver.com/mnews/article/366/000...,NaN
...,...,...,...,...
11319,2025-01-02,"""당장 돈 안되고 성능향상 기대 못 미쳐도""… 세계는 AI인프라 `영끌`",https://n.news.naver.com/mnews/article/029/000...,NaN
11320,2025-01-02,트럼프 2.0시대 중국…미 동맹국 이탈 기대하며 버티기[다시 만난 트럼프②],https://n.news.naver.com/mnews/article/032/000...,NaN
11321,2025-01-02,'美 51번째 주' 모욕 당했는데…트럼프에 찍소리 못하는 이유 [김리안의 에네르기파...,https://n.news.naver.com/mnews/article/015/000...,NaN
11322,2025-01-02,[산업은행] 2025년 상반기 KDB NextONE 참여 스타트업 모집(서울/부산)...,https://fintech.or.kr/web/board/boardContentsV...,NaN


In [65]:
# !pip install sqlalchemy pymysql

In [72]:
# !pip install python-dotenv

In [28]:
from dbio3 import to_db, load_data

In [29]:
df3 = load_data("fintech_news", "news_articles")
df3

OperationalError: (pymysql.err.OperationalError) (2003, "Can't connect to MySQL server on 'localhost' ([Errno 111] Connection refused)")
(Background on this error at: https://sqlalche.me/e/20/e3q8)